# Unified Dataset Transformation Workflow

This notebook merges and cleans the workflows from `datasets-merging.ipynb` and `transformer-500k.ipynb`.

Goal: document how raw multi-source text datasets are transformed into age-labeled parquet datasets for the project (age-group text classification).


## What This Notebook Covers

1. Common setup and reusable helpers.
2. Reddit (ABCDE) TSV -> cleaned parquet + balanced subsets.
3. Twitter/X (ABCDE city + country) TSV -> cleaned parquet + merged dataset.
4. Blog Authorship merged parquet -> inspection and examples.
5. PAN13 / PAN15 / PAN16 / PAN19 saved parquets -> quick EDA.
6. Hippocorpus CSV -> age-binned long-text parquet.
7. Final sanity checks and project-level summary.


In [2]:
from pathlib import Path
import math

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.auto import tqdm

AGE_BINS = [6, 13, 18, 30, 50, 65, float("inf")]
AGE_LABELS = ["6-12", "13-17", "18-29", "30-49", "50-64", "65+"]
MODEL_CLASSES = ["13-17", "18-29", "30-49", "50-64", "65+"]
SEED = 0.42


In [3]:
def first_existing_path(candidates):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p
    raise FileNotFoundError(f"None of these paths exists: {candidates}")


def ensure_parent_dir(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)


def age_distribution_parquet(parquet_path):
    dataset = ds.dataset(str(parquet_path), format="parquet")
    table = dataset.to_table(columns=["age"])
    vc = pc.value_counts(table["age"])

    counts = pd.DataFrame({
        "age": vc.field("values").to_pandas(),
        "count": vc.field("counts").to_pandas(),
    }).sort_values("count", ascending=False)
    counts["percent"] = (counts["count"] / counts["count"].sum() * 100).round(2)
    return counts.reset_index(drop=True)


def process_tsv_to_parquet(
    input_path,
    output_path,
    text_builder,
    age_col="DMGAgeAtPost",
    usecols=None,
    chunk_size=100_000,
):
    input_path = Path(input_path)
    output_path = Path(output_path)
    ensure_parent_dir(output_path)

    total_lines = sum(1 for _ in open(input_path, "r", encoding="utf-8")) - 1
    total_chunks = math.ceil(total_lines / chunk_size)

    writer = None
    reader = pd.read_csv(input_path, sep="	", chunksize=chunk_size, usecols=usecols)

    for chunk in tqdm(reader, total=total_chunks, desc=f"Processing {input_path.name}"):
        chunk["text"] = text_builder(chunk)
        age_num = pd.to_numeric(chunk[age_col], errors="coerce")
        chunk["age"] = pd.cut(age_num, bins=AGE_BINS, labels=AGE_LABELS, right=False)

        chunk = chunk[["text", "age"]].dropna(subset=["age"])
        chunk["text"] = chunk["text"].fillna("").astype(str)
        chunk["age"] = chunk["age"].astype(str)

        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(str(output_path), table.schema, compression="snappy")
        writer.write_table(table)

    if writer is not None:
        writer.close()

    return output_path


## 1) Reddit (ABCDE): TSV -> cleaned parquet

In [ ]:
reddit_raw = first_existing_path([
    "data/abcde/reddit/reddit_users_posts.tsv",
    "data/reddit/reddit_users_posts.tsv",
])

reddit_processed = Path("data/abcde/reddit/other/reddit_processed.parquet")

reddit_processed = process_tsv_to_parquet(
    input_path=reddit_raw,
    output_path=reddit_processed,
    usecols=["PostTitle", "PostSelftext", "DMGAgeAtPost"],
    text_builder=lambda c: c["PostTitle"].fillna("").astype(str) + ". " + c["PostSelftext"].fillna("").astype(str),
    age_col="DMGAgeAtPost",
    chunk_size=100_000,
)

print("Saved:", reddit_processed)
age_distribution_parquet(reddit_processed)


## 2) Reddit balanced subsets (from transformer workflow)

In [ ]:
con = duckdb.connect(database=":memory:")
con.execute(f"SELECT setseed({SEED})")

reddit_100k_per_class = reddit_processed.parent / "reddit_processed_100k_per_class.parquet"
reddit_200k_per_class = reddit_processed.parent / "reddit_processed_200k_per_class.parquet"

for per_class, out_path in [(100_000, reddit_100k_per_class), (200_000, reddit_200k_per_class)]:
    query = f"""
    COPY (
      SELECT age, text
      FROM (
        SELECT
          age,
          text,
          ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
        FROM read_parquet('{reddit_processed}')
        WHERE age IN ({', '.join(f"'{c}'" for c in MODEL_CLASSES)})
          AND text IS NOT NULL
          AND length(text) > 0
      )
      WHERE rn <= {per_class}
    )
    TO '{out_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """
    con.execute(f"SELECT setseed({SEED})")
    con.execute(query)
    print("Saved:", out_path)

con.close()


In [ ]:
print("100k/class distribution")
age_distribution_parquet(reddit_100k_per_class)


## 3) Reddit learning-curve subsets (1k -> 500k)

In [ ]:
sizes = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000]
output_dir = Path("data/abcde/reddit")
output_dir.mkdir(parents=True, exist_ok=True)

max_per_class = max(sizes) // len(MODEL_CLASSES)

con = duckdb.connect(database=":memory:")
con.execute(f"SELECT setseed({SEED})")

con.execute(f"""
    CREATE TABLE reddit_pool AS
    SELECT text, age
    FROM (
        SELECT text, age,
               ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
        FROM read_parquet('{reddit_processed}')
        WHERE age IN ({', '.join(f"'{c}'" for c in MODEL_CLASSES)})
    )
    WHERE rn <= {max_per_class}
""")

for size in sizes:
    per_class = size // len(MODEL_CLASSES)
    con.execute(f"SELECT setseed({SEED})")
    df = con.execute(f"""
        SELECT text, age
        FROM (
            SELECT text, age,
                   ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
            FROM reddit_pool
        )
        WHERE rn <= {per_class}
        ORDER BY RANDOM()
    """).df()

    out_path = output_dir / f"reddit_{size // 1000}k.parquet"
    df.to_parquet(out_path, index=False)
    print(f"Saved {out_path} ({len(df):,} rows)")

con.close()


## 4) Twitter/X (ABCDE): city + country TSV -> cleaned parquet -> merged parquet

In [ ]:
twitter_city_raw = first_existing_path([
    "data/abcde/twitter/other/city_user_posts.tsv",
    "data/abcde/tusc/city_user_posts.tsv",
])

twitter_country_raw = first_existing_path([
    "data/abcde/twitter/other/country_user_posts.tsv",
    "data/abcde/tusc/country_user_posts.tsv",
])

twitter_city_parquet = Path("data/abcde/twitter/other/tusc_processed.parquet")
twitter_country_parquet = Path("data/abcde/twitter/other/country_user_posts.parquet")
twitter_merged_parquet = Path("data/abcde/twitter/other/tusc_merged.parquet")

process_tsv_to_parquet(
    input_path=twitter_city_raw,
    output_path=twitter_city_parquet,
    usecols=["PostText", "DMGAgeAtPost"],
    text_builder=lambda c: c["PostText"],
    age_col="DMGAgeAtPost",
    chunk_size=100_000,
)

process_tsv_to_parquet(
    input_path=twitter_country_raw,
    output_path=twitter_country_parquet,
    usecols=["PostText", "DMGAgeAtPost"],
    text_builder=lambda c: c["PostText"],
    age_col="DMGAgeAtPost",
    chunk_size=100_000,
)

twitter_city_df = pd.read_parquet(twitter_city_parquet)
twitter_country_df = pd.read_parquet(twitter_country_parquet)

twitter_merged_df = pd.concat([twitter_city_df, twitter_country_df], ignore_index=True)
twitter_merged_df.to_parquet(twitter_merged_parquet, index=False)

print("Saved:", twitter_merged_parquet)
age_distribution_parquet(twitter_merged_parquet)


In [ ]:
con = duckdb.connect(database=":memory:")
con.execute(f"SELECT setseed({SEED})")

count_13_17 = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{twitter_merged_parquet}') WHERE age = '13-17'
""").fetchone()[0]

per_other = max((500_000 - count_13_17) // 4, 0)
twitter_500k = Path("data/abcde/twitter/twitter_500k.parquet")

query = f"""
COPY (
    SELECT text, age
    FROM (
        SELECT text, age,
               ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
        FROM read_parquet('{twitter_merged_parquet}')
        WHERE age IN ({', '.join(f"'{c}'" for c in MODEL_CLASSES)})
    )
    WHERE (age = '13-17')
       OR (age != '13-17' AND rn <= {per_other})
    ORDER BY RANDOM()
)
TO '{twitter_500k}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(query)
con.close()

print("Saved:", twitter_500k)
age_distribution_parquet(twitter_500k)


## 5) Blog Authorship: inspect merged blog parquet

In [10]:
blog_path = Path("data/blog/blog.parquet")
blog_df = pd.read_parquet(blog_path).copy()

blog_df["text"] = blog_df["text"].fillna("").astype(str)
blog_df["age"] = blog_df["age"].astype(str)
blog_df = blog_df[blog_df["age"].isin(MODEL_CLASSES)].reset_index(drop=True)

blog_df["char_len"] = blog_df["text"].str.len()
blog_df["word_len"] = blog_df["text"].str.split().map(len)
blog_df["preview"] = blog_df["text"].str.replace(r"\s+", " ", regex=True).str.slice(0, 180)

present_classes = [c for c in MODEL_CLASSES if c in set(blog_df["age"])]

print("Loaded:", blog_path)
print(f"Rows: {len(blog_df):,}")
print(f"Empty texts: {(blog_df['char_len'] == 0).sum():,}")
print("Classes:", ", ".join(present_classes))

blog_df[["age", "char_len", "word_len", "preview"]].head()

Loaded: data/blog/blog.parquet
Rows: 681,284
Empty texts: 0
Classes: 13-17, 18-29, 30-49


,age,char_len,word_len,preview
0,13-17,489,73,In het kader van kernfusie op aarde: MAAK JE ...
1,30-49,609,102,I had an interesting conversation with my Dad...
2,30-49,595,98,"If anything, Korea is a country of extremes. ..."
3,30-49,551,90,Take a read of this news article from urlLink...
4,30-49,555,91,"Ah, the Korean language...it looks so difficu..."


In [11]:
from IPython.display import display

class_dist = blog_df["age"].value_counts().rename_axis("age").reset_index(name="count")
class_dist["age"] = pd.Categorical(class_dist["age"], categories=present_classes, ordered=True)
class_dist["percent"] = (class_dist["count"] / class_dist["count"].sum() * 100).round(2)
class_dist = class_dist.sort_values("age").reset_index(drop=True)

length_summary = (
    blog_df.groupby("age")
    .agg(
        count=("text", "size"),
        avg_chars=("char_len", "mean"),
        median_chars=("char_len", "median"),
        p90_chars=("char_len", lambda s: s.quantile(0.90)),
        max_chars=("char_len", "max"),
        avg_words=("word_len", "mean"),
        median_words=("word_len", "median"),
    )
    .round(2)
    .reset_index()
)
length_summary["age"] = pd.Categorical(length_summary["age"], categories=present_classes, ordered=True)
length_summary = length_summary.sort_values("age").reset_index(drop=True)

example_frames = []
for age in present_classes:
    sample_n = min(3, int((blog_df["age"] == age).sum()))
    sample = (
        blog_df.loc[blog_df["age"] == age, ["age", "char_len", "word_len", "preview"]]
        .sample(n=sample_n, random_state=42)
        .sort_values("char_len", ascending=False)
    )
    example_frames.append(sample)

examples_per_class = pd.concat(example_frames, ignore_index=True)

print("Class distribution")
display(class_dist)

print("Length summary by class")
display(length_summary)

print("Sample examples by class")
display(examples_per_class)

Class distribution


,age,count,percent
0,13-17,235867,34.62
1,18-29,321447,47.18
2,30-49,123970,18.20


Length summary by class


,age,count,avg_chars,median_chars,p90_chars,max_chars,avg_words,median_words
0,13-17,235867,379.94,448.0,557.0,46136,67.95,80.0
1,18-29,321447,399.98,488.0,584.0,1826,70.25,87.0
2,30-49,123970,403.01,491.0,597.0,1721,69.53,86.0


Sample examples by class


,age,char_len,word_len,preview
0,13-17,547,101,So I got my official letter from Baylor LIFE ...
1,13-17,476,83,"Today sucked. I wanted to call Kelly, but nev..."
2,13-17,284,49,"i wonder, is there a difference between lovin..."
3,18-29,492,94,The Clock Part 1: The Naked Dance Of Twilight...
4,18-29,485,93,MY NOTEBOOK KANA VIRUS!!!!!!!! that's y there...
5,18-29,251,34,School girl got suspended for showing off her...
6,30-49,644,98,AlphaCAM Insights is a quick view into the wo...
7,30-49,602,107,Barry Bonds is on pace to not only break the ...
8,30-49,558,105,"Yep, went down to the Ice Palace (yeah, that'..."


### PAN datasets: quick EDA of saved document-level parquets

This subsection inspects the saved `pan13`, `pan15`, `pan16`, and `pan19` parquet files without rebuilding them. It reports dataset size, class imbalance, simple text-length statistics, and a couple of text previews per class.


In [6]:

from IPython.display import display

PAN_PARQUETS = {
    "pan13": Path("data/pan13/pan13.parquet"),
    "pan15": Path("data/pan15/pan15.parquet"),
    "pan16": Path("data/pan16/pan16.parquet")
}

PAN_NOTES = {
    "pan13": "English only; one row per <conversation>; mapped 10s/20s/30s to target bins.",
    "pan15": "English only; one row per <document>; kept only safely mappable age bins.",
    "pan16": "Empty placeholder: this local PAN16 archive contains tweet URLs/IDs instead of tweet text.",
    "pan19": "Celebrity profiling; one tweet per row; age derived conservatively from birthyear relative to 2019.",
}


def sql_quote(value):
    return str(value).replace("'", "''")


def sql_age_order(column="age"):
    parts = [f"WHEN {column} = '{age}' THEN {idx}" for idx, age in enumerate(MODEL_CLASSES)]
    return "CASE " + " ".join(parts) + " ELSE 999 END"


def pan_quick_eda(dataset_name, parquet_path, sample_per_class=2):
    parquet_sql = sql_quote(parquet_path.as_posix())
    con = duckdb.connect(database=":memory:")

    class_df = con.execute(
        f"""
        SELECT
            age,
            COUNT(*) AS count,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percent,
            ROUND(AVG(LENGTH(text)), 2) AS avg_chars,
            MAX(LENGTH(text)) AS max_chars,
            SUM(CASE WHEN LENGTH(TRIM(text)) = 0 THEN 1 ELSE 0 END) AS empty_texts
        FROM read_parquet('{parquet_sql}')
        GROUP BY age
        ORDER BY {sql_age_order('age')}
        """
    ).fetchdf()

    meta_df = pd.DataFrame([
        {
            "dataset": dataset_name,
            "rows": pq.ParquetFile(parquet_path).metadata.num_rows,
            "present_classes": len(class_df),
            "largest_class_percent": round(class_df["percent"].max(), 2),
            "file_size_mb": round(parquet_path.stat().st_size / (1024 * 1024), 2),
            "note": PAN_NOTES.get(dataset_name, ""),
        }
    ])

    example_frames = []
    for age in class_df["age"].tolist():
        example_df = con.execute(
            f"""
            SELECT
                age,
                LENGTH(text) AS char_len,
                LEFT(REGEXP_REPLACE(text, '\\s+', ' ', 'g'), 180) AS preview
            FROM read_parquet('{parquet_sql}')
            WHERE age = '{sql_quote(age)}'
              AND LENGTH(TRIM(text)) > 0
            LIMIT {sample_per_class}
            """
        ).fetchdf()
        if not example_df.empty:
            example_frames.append(example_df)

    examples_df = (
        pd.concat(example_frames, ignore_index=True)
        if example_frames
        else pd.DataFrame(columns=["age", "char_len", "preview"])
    )
    return meta_df, class_df, examples_df


for dataset_name, parquet_path in PAN_PARQUETS.items():
    print("=" * 100)
    print(f"{dataset_name.upper()} :: {parquet_path}")

    if not parquet_path.exists():
        print("Missing file.")
        continue

    row_count = pq.ParquetFile(parquet_path).metadata.num_rows
    if row_count == 0:
        empty_meta = pd.DataFrame([
            {
                "dataset": dataset_name,
                "rows": 0,
                "present_classes": 0,
                "largest_class_percent": None,
                "file_size_mb": round(parquet_path.stat().st_size / (1024 * 1024), 4),
                "note": PAN_NOTES.get(dataset_name, ""),
            }
        ])
        display(empty_meta)
        continue

    meta_df, class_df, examples_df = pan_quick_eda(dataset_name, parquet_path)
    display(meta_df)
    display(class_df)
    display(examples_df)


PAN13 :: data/pan13/pan13.parquet


,dataset,rows,present_classes,largest_class_percent,file_size_mb,note
0,pan13,413555,3,55.63,109.67,English only; one row per <conversation>; mapp...


,age,count,percent,avg_chars,max_chars,empty_texts
0,13-17,27651,6.69,489.31,986,0.0
1,18-29,155853,37.69,411.09,2883,0.0
2,30-49,230051,55.63,473.35,1520,0.0


,age,char_len,preview
0,13-17,195,"Hiya, I'm 11years old and board !! I go to Hig..."
1,13-17,88,"<img class=""smiley"" src=""http://www.pan.net/sm..."
2,18-29,570,The Spyware Doctor 2010 is the excellent spywa...
3,18-29,579,</br>; <br />;<p>;Wireless Television for comp...
4,30-49,510,The fantasy world of Spanish Fairy tales and f...
5,30-49,572,"Nevertheless, there are benefits about studyin..."


PAN15 :: data/pan15/pan15.parquet


,dataset,rows,present_classes,largest_class_percent,file_size_mb,note
0,pan15,7453,2,74.98,0.4,English only; one row per <document>; kept onl...


,age,count,percent,avg_chars,max_chars,empty_texts
0,18-29,5588,74.98,61.15,163,0.0
1,30-49,1865,25.02,87.16,146,0.0


,age,char_len,preview
0,18-29,124,I sit with a boy in english he became my frien...
1,18-29,42,boooored out of my mind #wannasleepbutdont
2,30-49,51,@username that tweet was mainly for you ;-) ch...
3,30-49,70,"I am watching Person of Interest, Pilot (S01E0..."


PAN16 :: data/pan16/pan16.parquet
Missing file.


## 6) Hippocorpus: CSV -> age-binned long-text parquet

In [ ]:
hippo_csv = first_existing_path(["hippoCorpusV2.csv", "data/hippocorpus/hippoCorpusV2.csv"])
hippo_df = pd.read_csv(hippo_csv, usecols=["story", "annotatorAge"])
hippo_df = hippo_df.rename(columns={"story": "text", "annotatorAge": "age"})

hippo_df["age"] = pd.cut(pd.to_numeric(hippo_df["age"], errors="coerce"), bins=AGE_BINS, labels=AGE_LABELS, right=False)
hippo_df = hippo_df[["text", "age"]].dropna(subset=["age"]).copy()
hippo_df["text"] = hippo_df["text"].fillna("").astype(str)
hippo_df["age"] = hippo_df["age"].astype(str)

hippo_df["token_len"] = hippo_df["text"].str.split().map(len)
hippo_long_df = hippo_df.loc[hippo_df["token_len"] > 128, ["text", "age"]]

hippo_long_path = Path("data/hippocorpus/hippocorpus_long.parquet")
ensure_parent_dir(hippo_long_path)
hippo_long_df.to_parquet(hippo_long_path, index=False)

print("Saved:", hippo_long_path)
age_distribution_parquet(hippo_long_path)


## 7) Final summary table for transformed outputs

In [ ]:
final_outputs = {
    "reddit_processed": Path("data/abcde/reddit/other/reddit_processed.parquet"),
    "reddit_100k_per_class": Path("data/abcde/reddit/other/reddit_processed_100k_per_class.parquet"),
    "reddit_200k_per_class": Path("data/abcde/reddit/other/reddit_processed_200k_per_class.parquet"),
    "twitter_merged": Path("data/abcde/twitter/other/tusc_merged.parquet"),
    "twitter_500k": Path("data/abcde/twitter/twitter_500k.parquet"),
    "blog_long": Path("data/blog/blog_long.parquet"),
    "blog_short_medium": Path("data/blog/blog_short_medium.parquet"),
    "hippocorpus_long": Path("data/hippocorpus/hippocorpus_long.parquet"),
}

rows = []
for name, path in final_outputs.items():
    if path.exists():
        try:
            row_count = duckdb.query(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
            rows.append({"dataset": name, "path": str(path), "rows": row_count})
        except Exception as e:
            rows.append({"dataset": name, "path": str(path), "rows": f"error: {e}"})
    else:
        rows.append({"dataset": name, "path": str(path), "rows": "missing"})

pd.DataFrame(rows).sort_values("dataset").reset_index(drop=True)


## Notes on Corrections Applied

- Removed notebook-only `pip install ...` cell from the workflow.
- Standardized path handling and added fallbacks for legacy folder layouts.
- Added helper functions to avoid repeated copy-paste transformations.
- Added explicit parent-directory creation before writing parquet outputs.
- Kept deterministic sampling with a fixed random seed.
- Added final sanity checks so the notebook clearly demonstrates project-ready transformed datasets.


## 8) Dataset standardization to 5-class target

Draft, merge-ready standardization for `blog`, `hippocorpus`, and `pan14` into a single schema for downstream 5-class age prediction.


### Shared utilities: loaders, schema, mapping stub, and output persistence

In [4]:
import json
import warnings
import xml.etree.ElementTree as ET

STANDARDIZED_BASE_DIR = Path("./outputs/standardized")
STANDARDIZED_BASE_DIR.mkdir(parents=True, exist_ok=True)

UNIFIED_SCHEMA_COLUMNS = ["text", "label", "label_name", "source", "doc_id", "meta"]

# Configurable mapping stubs. Keep empty until class definitions are finalized.
# Expected value format: {raw_label: (label_int_0_to_4, label_name)}
LABEL_MAPPING_CONFIG = {
    "blog": {
        # TODO: define blog raw label -> 5-class mapping
        # Example: "13-17": (0, "13-17"),
    },
    "hippocorpus": {
        # TODO: define hippocorpus raw label -> 5-class mapping
    },
    "pan14": {
        # TODO: define PAN14 raw label -> 5-class mapping
        # Example: "18-24": (1, "18-29"),
    },
}


def map_to_5_classes(raw_label, *, dataset_name):
    """Map source-specific raw labels to the unified 5-class target.

    Returns:
        tuple[int, str] | None
            (label, label_name) if mapping is configured, otherwise None.
    """
    # TODO: inspect label distribution / age ranges for each dataset.
    # TODO: define bin edges / class definitions for exactly 5 target classes.
    # TODO: verify no leakage / consistent interpretation across all datasets.
    if raw_label is None or pd.isna(raw_label):
        return None

    rules = LABEL_MAPPING_CONFIG.get(dataset_name, {})
    key = str(raw_label).strip()
    return rules.get(key)


def _first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def _safe_json_value(value):
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, (str, int, float, bool)) or value is None:
        return value

    if hasattr(value, "isoformat"):
        try:
            return value.isoformat()
        except Exception:
            pass

    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass

    return str(value)


def _meta_json_from_row(row, columns):
    payload = {}
    for col in columns:
        val = _safe_json_value(row[col])
        if val is not None:
            payload[col] = val
    return json.dumps(payload, ensure_ascii=False)


def read_tabular_file(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    if suffix in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t", low_memory=False)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        try:
            return pd.read_json(path, lines=True)
        except ValueError:
            return pd.read_json(path)

    raise ValueError(f"Unsupported file format: {path}")


def load_tabular_files(dataset_dir, *, preferred_globs):
    dataset_dir = Path(dataset_dir)
    files = []
    for pattern in preferred_globs:
        files.extend(dataset_dir.rglob(pattern))

    files = sorted({p.resolve() for p in files if p.is_file()})
    if not files:
        raise FileNotFoundError(f"No matching files found under {dataset_dir}")

    frames = []
    for path in files:
        try:
            frame = read_tabular_file(path)
            frame["__source_file"] = str(path)
            frames.append(frame)
        except Exception as exc:
            warnings.warn(f"Skipping unreadable file {path}: {exc}")

    if not frames:
        raise RuntimeError(f"No readable files found under {dataset_dir}")

    return pd.concat(frames, ignore_index=True)


def load_blog_raw(dataset_dir):
    return load_tabular_files(
        dataset_dir,
        preferred_globs=[
            "blogtext.csv",
            "*.csv",
            "*.parquet",
            "*.tsv",
            "*.jsonl",
            "*.json",
        ],
    )


def load_hippocorpus_raw(dataset_dir):
    return load_tabular_files(
        dataset_dir,
        preferred_globs=[
            "hippoCorpusV2.csv",
            "*.csv",
            "*.parquet",
            "*.tsv",
            "*.jsonl",
            "*.json",
        ],
    )


def extract_pan14_text(xml_path):
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError as exc:
        warnings.warn(f"XML parse failed for {xml_path}: {exc}")
        return ""

    parts = []
    for doc in root.findall(".//document"):
        text = "".join(doc.itertext()).strip()
        if text:
            parts.append(text)
    return "\n\n".join(parts)


def load_pan14_raw(dataset_dir):
    dataset_dir = Path(dataset_dir)
    truth_files = sorted(dataset_dir.rglob("truth.txt"))
    if not truth_files:
        raise FileNotFoundError(f"No truth.txt files found under {dataset_dir}")

    records = []
    for truth_path in truth_files:
        corpus_root = truth_path.parent
        xml_by_author = {p.stem: p for p in corpus_root.glob("*.xml")}
        if not xml_by_author:
            xml_by_author = {p.stem: p for p in corpus_root.rglob("*.xml")}

        with open(truth_path, "r", encoding="utf-8", errors="ignore") as fh:
            for line_no, line in enumerate(fh, start=1):
                line = line.strip()
                if not line:
                    continue

                parts = line.split(":::")
                if len(parts) < 3:
                    warnings.warn(f"Malformed truth row: {truth_path}:{line_no} -> {line}")
                    continue

                author_id, gender, raw_label = parts[0].strip(), parts[1].strip(), parts[2].strip()
                xml_path = xml_by_author.get(author_id)
                if xml_path is None:
                    warnings.warn(f"Missing XML for author {author_id} in {truth_path}")
                    continue

                text = extract_pan14_text(xml_path)
                records.append(
                    {
                        "doc_id": author_id,
                        "text": text,
                        "raw_label": raw_label,
                        "gender": gender,
                        "domain": corpus_root.name,
                        "truth_file": str(truth_path),
                        "xml_file": str(xml_path),
                    }
                )

    if not records:
        raise RuntimeError("PAN14 parsing produced 0 rows")

    return pd.DataFrame(records)


def standardize_to_unified_schema(
    raw_df,
    *,
    dataset_name,
    text_candidates,
    label_candidates,
    doc_id_candidates,
):
    raw_df = raw_df.copy()

    text_col = _first_existing_column(raw_df, text_candidates)
    if text_col is None:
        raise KeyError(f"{dataset_name}: no text column found in {text_candidates}")

    label_col = _first_existing_column(raw_df, label_candidates)
    doc_id_col = _first_existing_column(raw_df, doc_id_candidates)

    work = pd.DataFrame(index=raw_df.index)
    work["text"] = raw_df[text_col].fillna("").astype(str).str.strip()

    if label_col is None:
        work["raw_label"] = pd.NA
    else:
        work["raw_label"] = raw_df[label_col]

    if doc_id_col is None:
        if "__source_file" in raw_df.columns:
            source_stem = raw_df["__source_file"].map(lambda p: Path(p).stem)
            work["doc_id"] = source_stem + ":" + raw_df.index.astype(str)
        else:
            work["doc_id"] = f"{dataset_name}:" + raw_df.index.astype(str)
    else:
        work["doc_id"] = raw_df[doc_id_col].astype(str)

    reserved = {
        text_col,
        label_col,
        doc_id_col,
        "text",
        "label",
        "label_name",
        "source",
        "doc_id",
        "meta",
    }
    meta_cols = [c for c in raw_df.columns if c not in reserved and c is not None]

    if meta_cols:
        work["meta"] = raw_df.apply(lambda row: _meta_json_from_row(row, meta_cols), axis=1)
    else:
        work["meta"] = "{}"

    mapped = work["raw_label"].apply(lambda v: map_to_5_classes(v, dataset_name=dataset_name))
    work["label"] = mapped.apply(lambda x: x[0] if isinstance(x, tuple) else None)
    work["label_name"] = mapped.apply(lambda x: x[1] if isinstance(x, tuple) else None)

    mapped_mask = work["label"].notna()
    dropped = int((~mapped_mask).sum())
    if dropped:
        warnings.warn(
            f"{dataset_name}: dropped {dropped:,} rows because map_to_5_classes has no rule for their raw labels"
        )

    standardized = work.loc[mapped_mask, ["text", "label", "label_name", "doc_id", "meta"]].copy()
    standardized["label"] = standardized["label"].astype(int)
    standardized["source"] = dataset_name
    standardized = standardized[UNIFIED_SCHEMA_COLUMNS]
    standardized = standardized[standardized["text"].str.len() > 0].reset_index(drop=True)

    return standardized


def save_standardized_dataset(dataset_name, standardized_df, *, rows_loaded):
    output_dir = STANDARDIZED_BASE_DIR / dataset_name
    output_dir.mkdir(parents=True, exist_ok=True)

    parquet_path = output_dir / "data.parquet"
    csv_path = output_dir / "data.csv"
    stats_path = output_dir / "stats.json"

    standardized_df.to_parquet(parquet_path, index=False)
    standardized_df.to_csv(csv_path, index=False)

    class_counts = {
        str(int(k)): int(v)
        for k, v in standardized_df["label"].value_counts().sort_index().to_dict().items()
    } if not standardized_df.empty else {}

    stats = {
        "dataset": dataset_name,
        "total_rows_loaded": int(rows_loaded),
        "rows_mapped_to_5_classes": int(len(standardized_df)),
        "rows_dropped_unmapped": int(rows_loaded - len(standardized_df)),
        "class_counts": class_counts,
    }

    stats_path.write_text(json.dumps(stats, indent=2, ensure_ascii=False))
    return {
        "parquet": str(parquet_path),
        "csv": str(csv_path),
        "stats": str(stats_path),
    }, stats


### Blog dataset (draft loader + standardization)

In [ ]:
blog_dataset_dir = Path("/Users/georgijkutivadze/projects/mago-text-scoring/age/data/blog")

blog_raw_df = load_blog_raw(blog_dataset_dir)
blog_std_df = standardize_to_unified_schema(
    blog_raw_df,
    dataset_name="blog",
    text_candidates=["text", "content", "post", "body", "story"],
    label_candidates=["age", "age_group", "label", "class", "target"],
    doc_id_candidates=["doc_id", "id", "post_id", "uid", "user_id", "author_id"],
)

blog_paths, blog_stats = save_standardized_dataset(
    "blog",
    blog_std_df,
    rows_loaded=len(blog_raw_df),
)

print("Saved:", blog_paths)
print(json.dumps(blog_stats, indent=2, ensure_ascii=False))
blog_std_df.head()


### Hippocorpus dataset (draft loader + standardization)

In [ ]:
hippocorpus_dataset_dir = Path("/Users/georgijkutivadze/projects/mago-text-scoring/age/data/hippocorpus")

hippocorpus_raw_df = load_hippocorpus_raw(hippocorpus_dataset_dir)
hippocorpus_std_df = standardize_to_unified_schema(
    hippocorpus_raw_df,
    dataset_name="hippocorpus",
    text_candidates=["story", "text", "content", "body", "post"],
    label_candidates=["annotatorAge", "age", "age_group", "label", "class", "target"],
    doc_id_candidates=["doc_id", "id", "storyid", "uid", "user_id", "author_id"],
)

hippocorpus_paths, hippocorpus_stats = save_standardized_dataset(
    "hippocorpus",
    hippocorpus_std_df,
    rows_loaded=len(hippocorpus_raw_df),
)

print("Saved:", hippocorpus_paths)
print(json.dumps(hippocorpus_stats, indent=2, ensure_ascii=False))
hippocorpus_std_df.head()


### PAN14 dataset (draft loader + standardization)

In [ ]:
pan14_dataset_dir = Path("/Users/georgijkutivadze/projects/mago-text-scoring/age/data/pan14")

pan14_raw_df = load_pan14_raw(pan14_dataset_dir)
pan14_std_df = standardize_to_unified_schema(
    pan14_raw_df,
    dataset_name="pan14",
    text_candidates=["text", "document", "content", "body"],
    label_candidates=["raw_label", "age_group", "age", "label", "class", "target"],
    doc_id_candidates=["doc_id", "author_id", "id", "uid"],
)

pan14_paths, pan14_stats = save_standardized_dataset(
    "pan14",
    pan14_std_df,
    rows_loaded=len(pan14_raw_df),
)

print("Saved:", pan14_paths)
print(json.dumps(pan14_stats, indent=2, ensure_ascii=False))
pan14_std_df.head()


### Sanity checks and merge

In [ ]:
standardized_frames = []
for dataset_name in ["blog", "hippocorpus", "pan14"]:
    parquet_path = STANDARDIZED_BASE_DIR / dataset_name / "data.parquet"
    csv_path = STANDARDIZED_BASE_DIR / dataset_name / "data.csv"

    if parquet_path.exists():
        df = pd.read_parquet(parquet_path)
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        warnings.warn(f"Missing standardized output for {dataset_name}")
        continue

    missing_cols = [c for c in UNIFIED_SCHEMA_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{dataset_name}: missing required columns {missing_cols}")

    standardized_frames.append(df[UNIFIED_SCHEMA_COLUMNS].copy())

if standardized_frames:
    all_merged_df = pd.concat(standardized_frames, ignore_index=True)
else:
    all_merged_df = pd.DataFrame(columns=UNIFIED_SCHEMA_COLUMNS)

print("Rows by source:")
if all_merged_df.empty:
    print("Merged dataframe is empty (expected until mapping rules are configured).")
else:
    print(all_merged_df.groupby("source").size().sort_values(ascending=False))

    print("\nClass distribution per source (label):")
    print(pd.crosstab(all_merged_df["source"], all_merged_df["label"], margins=True))

    print("\nClass distribution overall (label_name):")
    print(all_merged_df["label_name"].value_counts(dropna=False))

all_merged_path = STANDARDIZED_BASE_DIR / "all_merged.parquet"
all_merged_df.to_parquet(all_merged_path, index=False)
print("\nSaved merged output:", all_merged_path)

all_merged_df.head()
